In [1]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from functools import reduce

In [2]:
# ============================================================
# 1. SPARK SESSION (configured for local run against S3)
# ============================================================

import os
from pathlib import Path
from dotenv import load_dotenv

env_file = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_file, override=True)

AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION", "eu-north-1")

HADOOP_VERSION = "3.4.1"
AWS_SDK_VERSION = "1.12.767"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

spark = (
    SparkSession.builder
    .appName("Ecommerce-Historical-API-ETL")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        f"org.apache.hadoop:hadoop-aws:{HADOOP_VERSION},com.amazonaws:aws-java-sdk-bundle:{AWS_SDK_VERSION}",
    )
    .config("spark.hadoop.fs.s3a.connection.ttl", "86400000")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", f"s3.{AWS_REGION}.amazonaws.com")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "bytebuffer")
    .getOrCreate()
)

print("Spark session ready.")


Spark session ready.


In [3]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

BUCKET = "yassin-s3-bucket-161828268081-eu-north-1-an"

# Historical data
RAW_PATH = f"s3a://{BUCKET}/raw/ecommerce"

# API incremental data (individual order_<id>.json files, written by the
# ingestion step -- see run_ingestion_local.py)
API_RAW_PATH = f"s3a://{BUCKET}/api_raw/orders"

# Curated / processed data -- matches the assignment's required S3 layout
PROCESSED_PATH = f"s3a://{BUCKET}/processed/ecommerce"


In [ ]:
# # ============================================================
# # REDSHIFT CONFIGURATION
# # ============================================================

# REDSHIFT_CONNECTION = "ecommerce-redshift-connection"

# REDSHIFT_DATABASE = "dev"

# REDSHIFT_SCHEMA = "public"

# # S3 temporary directory used by Spark/Glue
# REDSHIFT_TEMP_DIR = (
#     f"s3://{BUCKET}/redshift-temp/"
# )

In [4]:
# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def clean_column_names(df):

    for column in df.columns:

        new_name = (
            column.strip()
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        df = df.withColumnRenamed(
            column,
            new_name
        )

    return df


def read_csv(filename):

    path = f"{RAW_PATH}/{filename}"

    print("Reading:", path)

    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .csv(path)
    )


def write_parquet(df, table_name):

    output_path = (
        f"{PROCESSED_PATH}/{table_name}"
    )

    print(
        f"Writing {table_name} -> {output_path}"
    )

    (
        df.write
        .mode("overwrite")
        .format("parquet")
        .option("compression", "snappy")
        .save(output_path)
    )

In [5]:
# ============================================================
# 4. READ HISTORICAL DATA
# ============================================================

print("============================================")
print("READING HISTORICAL DATA")
print("============================================")


customers = read_csv("customers.csv")
categories = read_csv("categories.csv")
products = read_csv("products.csv")
departments = read_csv("departments.csv")
employees = read_csv("employees.csv")
suppliers = read_csv("suppliers.csv")
orders = read_csv("orders.csv")
order_details = read_csv("order_details.csv")
payments = read_csv("payments.csv")
product_suppliers = read_csv(
    "product_suppliers.csv"
)
shippers = read_csv("shippers.csv")
shipments = read_csv("shipments.csv")

READING HISTORICAL DATA
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/customers.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/categories.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/products.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/departments.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/employees.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/suppliers.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/orders.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/order_details.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/payments.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/product_suppliers.csv
Reading: s3a://yassin-s3-bucket-161828268081-eu-north-1-an/raw/ecommerce/shippers.csv
Reading: s

In [6]:
customers.printSchema()                  
categories.printSchema()
products.printSchema()
departments.printSchema()
employees.printSchema()
suppliers.printSchema()
orders.printSchema()
order_details.printSchema()
payments.printSchema()
product_suppliers.printSchema()
shippers.printSchema()
shipments.printSchema()

root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- RegistrationDate: date (nullable = true)

root
 |-- CategoryID: integer (nullable = true)
 |-- CategoryName: string (nullable = true)

root
 |-- ProductID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Price: double (nullable = true)
 |-- Cost: double (nullable = true)
 |-- Stock: integer (nullable = true)

root
 |-- DepartmentID: integer (nullable = true)
 |-- DepartmentName: string (nullable = true)

root
 |-- EmployeeID: integer (nullable = true)
 |-- ManagerID: integer (nullable = true)
 |-- DepartmentID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Salary: double (nul

In [7]:
# ============================================================
# 5. CLEAN COLUMN NAMES
# ============================================================

print("============================================")
print("CLEANING COLUMN NAMES")
print("============================================")


customers = clean_column_names(customers)

categories = clean_column_names(categories)

products = clean_column_names(products)

departments = clean_column_names(departments)

employees = clean_column_names(employees)

suppliers = clean_column_names(suppliers)

orders = clean_column_names(orders)

order_details = clean_column_names(order_details)

payments = clean_column_names(payments)

product_suppliers = clean_column_names(
    product_suppliers
)

shippers = clean_column_names(shippers)

shipments = clean_column_names(shipments)

CLEANING COLUMN NAMES


In [8]:
# ============================================================
# 6. READ API JSON
# ============================================================

print("============================================")
print("READING API JSON")
print("============================================")

api_json = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API JSON schema:")

api_json.printSchema()

READING API JSON
API JSON schema:
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |

In [9]:
import boto3

s3 = boto3.client("s3", region_name="eu-north-1")

response = s3.list_objects_v2(
    Bucket="yassin-s3-bucket-161828268081-eu-north-1-an",
    Prefix="api_raw/orders/"
)

for obj in response.get("Contents", []):
    print(obj["Key"])


api_raw/orders/order_2000001.json
api_raw/orders/order_2000002.json
api_raw/orders/order_2000003.json
api_raw/orders/order_2000004.json
api_raw/orders/order_2000005.json
api_raw/orders/order_2000006.json
api_raw/orders/order_2000007.json
api_raw/orders/order_2000008.json
api_raw/orders/order_2000009.json
api_raw/orders/order_2000010.json
api_raw/orders/order_2000011.json
api_raw/orders/order_2000012.json
api_raw/orders/order_2000013.json
api_raw/orders/order_2000014.json
api_raw/orders/order_2000015.json
api_raw/orders/order_2000016.json
api_raw/orders/order_2000017.json
api_raw/orders/order_2000018.json
api_raw/orders/order_2000019.json
api_raw/orders/order_2000020.json
api_raw/orders/order_2000021.json
api_raw/orders/order_2000022.json
api_raw/orders/order_2000023.json
api_raw/orders/order_2000024.json
api_raw/orders/order_2000025.json
api_raw/orders/order_2000026.json
api_raw/orders/order_2000027.json
api_raw/orders/order_2000028.json
api_raw/orders/order_2000029.json
api_raw/orders

In [10]:
# ============================================================
# 6. READ API JSON
# ============================================================

print("============================================")
print("READING API JSON")
print("============================================")

api_json = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API JSON schema:")

api_json.printSchema()

READING API JSON
API JSON schema:
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |

In [11]:
from pyspark.sql import functions as F

# BUCKET and API_RAW_PATH already defined in the CONFIGURATION cell above --
# reusing them here instead of redefining avoids the two ever drifting apart.

api_orders = (
    spark.read
    .option("multiLine", True)
    .json(API_RAW_PATH)
)

print("API records:", api_orders.count())

api_orders.printSchema()


API records: 300
root
 |-- metadata: struct (nullable = true)
 |    |-- event_id: string (nullable = true)
 |    |-- event_timestamp: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- source: string (nullable = true)
 |-- order: struct (nullable = true)
 |    |-- CustomerID: long (nullable = true)
 |    |-- OrderDate: string (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- Status: string (nullable = true)
 |-- order_details: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- Discount: double (nullable = true)
 |    |    |-- OrderDetailID: long (nullable = true)
 |    |    |-- OrderID: long (nullable = true)
 |    |    |-- ProductID: long (nullable = true)
 |    |    |-- Quantity: long (nullable = true)
 |    |    |-- UnitPrice: double (nullable = true)
 |-- payment: struct (nullable = true)
 |    |-- Amount: double (nullable = true)
 |    |-- OrderID: long (nullable = true)
 |    |-- PaymentDate: s

In [12]:
api_orders_df = api_orders.select(
    
    F.col("order.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("order.CustomerID")
        .cast("long")
        .alias("customerid"),

    F.to_date(
        F.col("order.OrderDate")
    ).alias("orderdate"),

    F.upper(
        F.trim(
            F.col("order.Status")
        )
    ).alias("status"),

    # API metadata
    F.col("metadata.event_id")
        .alias("event_id"),

    F.to_timestamp(
        F.col("metadata.event_timestamp")
    ).alias("event_timestamp"),

    F.col("metadata.event_type")
        .alias("event_type"),

    F.col("metadata.source")
        .alias("source")
)

api_orders_df.show(10, truncate=False)

+-------+----------+----------+---------+------------------------------------+--------------------------+-------------+------------+
|orderid|customerid|orderdate |status   |event_id                            |event_timestamp           |event_type   |source      |
+-------+----------+----------+---------+------------------------------------+--------------------------+-------------+------------+
|2000063|434201    |2026-09-23|CONFIRMED|d2045e20-704c-4367-a2ea-51aa600ce9db|2026-09-23 02:36:22.967273|ORDER_CREATED|ordering_api|
|2000180|111243    |2026-09-23|CONFIRMED|5013b375-a6bb-4b95-b9e3-e8345830e604|2026-09-23 02:36:23.147568|ORDER_CREATED|ordering_api|
|2000065|168318    |2026-09-23|CONFIRMED|5a33ba7f-caf8-4e1c-86c1-0d58f2f3f6f4|2026-09-23 02:36:22.969506|ORDER_CREATED|ordering_api|
|2000123|246971    |2026-09-23|CONFIRMED|a2e73ccb-075d-4c26-b34b-0b536a6e3c0a|2026-09-23 02:36:23.054895|ORDER_CREATED|ordering_api|
|2000214|461816    |2026-09-23|CONFIRMED|93811dc9-cd88-45c9-a951-5e1a

In [13]:
api_order_details_df = (
    api_orders
    .select(
        
        F.col("order.OrderID")
            .cast("long")
            .alias("orderid"),

        F.explode(
            F.col("order_details")
        ).alias("detail")
    )
)
api_order_details_df.show()

+-------+--------------------+
|orderid|              detail|
+-------+--------------------+
|2000063|{10.0, 5000149, 2...|
|2000063|{5.0, 5000150, 20...|
|2000063|{5.0, 5000151, 20...|
|2000063|{0.0, 5000152, 20...|
|2000180|{10.0, 5000442, 2...|
|2000180|{10.0, 5000443, 2...|
|2000180|{0.0, 5000444, 20...|
|2000180|{0.0, 5000445, 20...|
|2000065|{10.0, 5000157, 2...|
|2000065|{0.0, 5000158, 20...|
|2000065|{5.0, 5000159, 20...|
|2000065|{10.0, 5000160, 2...|
|2000123|{15.0, 5000302, 2...|
|2000123|{15.0, 5000303, 2...|
|2000123|{0.0, 5000304, 20...|
|2000123|{15.0, 5000305, 2...|
|2000214|{10.0, 5000519, 2...|
|2000214|{10.0, 5000520, 2...|
|2000214|{10.0, 5000521, 2...|
|2000214|{0.0, 5000522, 20...|
+-------+--------------------+
only showing top 20 rows


In [14]:
api_order_details_df = api_order_details_df.select(

    "orderid",

    F.col("detail.OrderDetailID")
        .cast("long")
        .alias("orderdetailid"),

    F.col("detail.ProductID")
        .cast("long")
        .alias("productid"),

    F.col("detail.Quantity")
        .cast("long")
        .alias("quantity"),

    F.col("detail.UnitPrice")
        .cast("decimal(12,2)")
        .alias("unitprice"),

    F.col("detail.Discount")
        .cast("decimal(12,2)")
        .alias("discount")
)
api_order_details_df.show()

+-------+-------------+---------+--------+---------+--------+
|orderid|orderdetailid|productid|quantity|unitprice|discount|
+-------+-------------+---------+--------+---------+--------+
|2000063|      5000149|     5987|       3|   861.96|   10.00|
|2000063|      5000150|     6327|       1|   103.17|    5.00|
|2000063|      5000151|     1037|       3|   913.00|    5.00|
|2000063|      5000152|     5142|       5|   943.92|    0.00|
|2000180|      5000442|     3385|       5|   630.54|   10.00|
|2000180|      5000443|     6562|       2|   968.05|   10.00|
|2000180|      5000444|     1142|       4|   351.08|    0.00|
|2000180|      5000445|     5624|       4|   283.17|    0.00|
|2000065|      5000157|     8965|       1|   893.25|   10.00|
|2000065|      5000158|     8138|       1|   881.94|    0.00|
|2000065|      5000159|     5862|       1|   622.46|    5.00|
|2000065|      5000160|     3029|       2|   715.77|   10.00|
|2000123|      5000302|     2514|       5|   235.94|   15.00|
|2000123

In [15]:
api_payments_df = api_orders.select(

    F.col("payment.PaymentID")
        .cast("long")
        .alias("paymentid"),

    F.col("payment.OrderID")
        .cast("long")
        .alias("orderid"),

    F.trim(
        F.col("payment.PaymentMethod")
    ).alias("paymentmethod"),

    F.to_date(
        F.col("payment.PaymentDate")
    ).alias("paymentdate"),

    F.col("payment.Amount")
        .cast("decimal(12,2)")
        .alias("amount"),

    F.upper(
        F.trim(
            F.col("payment.PaymentStatus")
        )
    ).alias("paymentstatus")
)

api_payments_df.show(10)

+---------+-------+----------------+-----------+--------+-------------+
|paymentid|orderid|   paymentmethod|paymentdate|  amount|paymentstatus|
+---------+-------+----------------+-----------+--------+-------------+
|  2000063|2000063|Cash on Delivery| 2026-09-23|11161.52|    COMPLETED|
|  2000180|2000180|Cash on Delivery| 2026-09-23| 8163.29|    COMPLETED|
|  2000065|2000065|Cash on Delivery| 2026-09-23| 4114.77|    COMPLETED|
|  2000123|2000123|Cash on Delivery| 2026-09-23| 6001.14|    COMPLETED|
|  2000214|2000214|   Bank Transfer| 2026-09-23| 9075.85|    COMPLETED|
|  2000032|2000032|Cash on Delivery| 2026-09-23| 6495.47|    COMPLETED|
|  2000073|2000073|Cash on Delivery| 2026-09-23| 7115.70|    COMPLETED|
|  2000113|2000113|Cash on Delivery| 2026-09-23| 2165.79|    COMPLETED|
|  2000202|2000202|Cash on Delivery| 2026-09-23| 6557.11|    COMPLETED|
|  2000257|2000257|Cash on Delivery| 2026-09-23| 4534.44|    COMPLETED|
+---------+-------+----------------+-----------+--------+-------

In [16]:
api_shipments_df = api_orders.select(

    F.col("shipment.ShipmentID")
        .cast("long")
        .alias("shipmentid"),

    F.col("shipment.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("shipment.ShipperID")
        .cast("long")
        .alias("shipperid"),

    F.to_date(
        F.col("shipment.ShipDate")
    ).alias("shipdate"),

    F.to_date(
        F.col("shipment.DeliveryDate")
    ).alias("deliverydate")
)

api_shipments_df.show(10)

+----------+-------+---------+----------+------------+
|shipmentid|orderid|shipperid|  shipdate|deliverydate|
+----------+-------+---------+----------+------------+
|    500061|2000063|      378|2026-09-24|  2026-09-26|
|    500175|2000180|      582|2026-09-24|  2026-09-29|
|    500063|2000065|      638|2026-09-24|  2026-09-26|
|    500120|2000123|       59|2026-09-24|  2026-09-26|
|    500208|2000214|      930|2026-09-24|  2026-09-26|
|    500032|2000032|      970|2026-09-24|  2026-09-28|
|    500071|2000073|      608|2026-09-24|  2026-09-28|
|    500110|2000113|      847|2026-09-24|  2026-09-27|
|    500196|2000202|      609|2026-09-24|  2026-09-26|
|    500250|2000257|      690|2026-09-24|  2026-09-26|
+----------+-------+---------+----------+------------+
only showing top 10 rows


In [17]:
api_order_summary_df = api_orders.select(

    F.col("order.OrderID")
        .cast("long")
        .alias("orderid"),

    F.col("summary.subtotal")
        .cast("decimal(12,2)")
        .alias("subtotal"),

    F.col("summary.discount")
        .cast("decimal(12,2)")
        .alias("discount"),

    F.col("summary.tax_rate")
        .cast("decimal(5,2)")
        .alias("tax_rate"),

    F.col("summary.tax_amount")
        .cast("decimal(12,2)")
        .alias("tax_amount"),

    F.col("summary.shipping_cost")
        .cast("decimal(12,2)")
        .alias("shipping_cost"),

    F.col("summary.total_amount")
        .cast("decimal(12,2)")
        .alias("total_amount")
)
api_order_summary_df.show(10)

+-------+--------+--------+--------+----------+-------------+------------+
|orderid|subtotal|discount|tax_rate|tax_amount|shipping_cost|total_amount|
+-------+--------+--------+--------+----------+-------------+------------+
|2000063| 9746.95|  400.70|    0.14|   1364.57|        50.00|    11161.52|
|2000180| 7116.92|  508.88|    0.14|    996.37|        50.00|     8163.29|
|2000065| 3565.59|  263.60|    0.14|    499.18|        50.00|     4114.77|
|2000123| 5220.30|  492.06|    0.14|    730.84|        50.00|     6001.14|
|2000214| 7917.41|  689.09|    0.14|   1108.44|        50.00|     9075.85|
|2000032| 5653.92|  559.58|    0.14|    791.55|        50.00|     6495.47|
|2000073| 6197.98|  427.71|    0.14|    867.72|        50.00|     7115.70|
|2000113| 1855.96|  197.41|    0.14|    259.83|        50.00|     2165.79|
|2000202| 5707.99|  502.47|    0.14|    799.12|        50.00|     6557.11|
|2000257| 3933.72|  131.61|    0.14|    550.72|        50.00|     4534.44|
+-------+--------+-------

In [18]:
historical_orders = orders.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
)

In [19]:
new_orders = api_orders_df.select(
    "orderid",
    "customerid",
    "orderdate",
    "status"
)

In [20]:
all_orders = (
    historical_orders
    .unionByName(new_orders)
    .dropDuplicates(["orderid"])
)
all_orders.show(10)

+-------+----------+----------+---------+
|orderid|customerid| orderdate|   status|
+-------+----------+----------+---------+
|      1|      8870|2024-05-03|Cancelled|
|      2|      2142|2026-01-18|  Pending|
|      3|      8030|2026-01-14|Delivered|
|      4|      4295|2025-06-05|Delivered|
|      5|      4623|2025-05-26|Delivered|
|      6|      9949|2024-09-01|  Shipped|
|      7|      5458|2025-11-01|Cancelled|
|      8|      5029|2024-08-09|Cancelled|
|      9|      3854|2025-01-16|Delivered|
|     10|      9903|2025-08-15|Cancelled|
+-------+----------+----------+---------+
only showing top 10 rows


In [21]:
print("Historical orders:", historical_orders.count())
print("API orders:", new_orders.count())
print("Combined orders:", all_orders.count())

Historical orders: 50000
API orders: 300
Combined orders: 50300


In [22]:
historical_orders_details = order_details.select(
    "OrderDetailID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "Discount"
)

In [23]:
new_orders_details = api_order_details_df.select(
    "OrderDetailID",
    "OrderID",
    "ProductID",
    "Quantity",
    "UnitPrice",
    "Discount"
)

In [24]:
all_orders_details = (
    historical_orders_details
    .unionByName(new_orders_details)
    .dropDuplicates(["OrderDetailID"])
)
all_orders_details.show(10)

+-------------+-------+---------+--------+---------+--------+
|OrderDetailID|OrderID|ProductID|Quantity|UnitPrice|Discount|
+-------------+-------+---------+--------+---------+--------+
|           26|  26004|      342|       5|  1253.86|    0.00|
|           29|  27162|      707|       3|   521.53|    0.00|
|          474|  45731|      267|       8|  2340.38|   10.00|
|          964|  13989|      172|       9|  2110.12|   10.00|
|         1677|  48400|      413|       8|  2534.22|    5.00|
|         1697|  30508|      824|       1|  2309.12|   15.00|
|         1806|  27067|      857|       9|   1547.7|    0.00|
|         1950|  18710|       91|       6|  2351.66|   10.00|
|         2040|  35516|      919|       7|  2171.69|    5.00|
|         2214|  26788|      496|       8|  2103.03|   20.00|
+-------------+-------+---------+--------+---------+--------+
only showing top 10 rows


In [25]:
print("Historical orders_details:", historical_orders_details.count())
print("API orders_details:", new_orders_details.count())
print("Combined orders_details:", all_orders_details.count())

Historical orders_details: 100000
API orders_details: 732
Combined orders_details: 100732


In [26]:
historical_payments = payments.select(
    "paymentid",
    "orderid",
    "paymentmethod",
    "paymentdate",
    "amount"
)

In [27]:
new_orders_payments = api_payments_df.select(
    "paymentid",
    "orderid",
    "paymentmethod",
    "paymentdate",
    "amount"
)

In [28]:
all_payments = (
    historical_payments
    .unionByName(new_orders_payments)
    .dropDuplicates(["paymentid"])
)
all_payments.show(10)

+---------+-------+-------------+-----------+-------+
|paymentid|orderid|paymentmethod|paymentdate| amount|
+---------+-------+-------------+-----------+-------+
|        1|   7144|Bank Transfer| 2026-03-30|1912.52|
|        2|  38187|       PayPal| 2025-05-12|2520.21|
|        3|  10641|  Credit Card| 2025-08-05| 609.92|
|        4|  44897|Bank Transfer| 2024-11-06|4654.04|
|        5|  45202|Bank Transfer| 2024-07-24| 200.19|
|        6|  15114|         Cash| 2024-06-07|   44.8|
|        7|  26717|         Cash| 2026-04-15|3849.76|
|        8|  27343|  Credit Card| 2024-03-09|1631.14|
|        9|  20619|         Cash| 2024-12-06|3859.07|
|       10|  27807|   Debit Card| 2025-10-28|1484.31|
+---------+-------+-------------+-----------+-------+
only showing top 10 rows


In [29]:
print("Historical payments:", historical_payments.count())
print("API payments:", new_orders_payments.count())
print("Combined payments:", all_payments.count())

Historical payments: 45000
API payments: 300
Combined payments: 45300


In [30]:
historical_shipments = shipments.select(
   "ShipmentID",
    "OrderID",
    "ShipperID",
    "ShipDate",
    "DeliveryDate"
)

In [31]:
new_orders_shipments = api_shipments_df.select(
   "ShipmentID",
    "OrderID",
    "ShipperID",
    "ShipDate",
    "DeliveryDate"
)

In [32]:
all_shipments = (
    historical_shipments
    .unionByName(new_orders_shipments)
    .dropDuplicates(["ShipmentID"])
)
all_shipments.show(10)

+----------+-------+---------+----------+------------+
|ShipmentID|OrderID|ShipperID|  ShipDate|DeliveryDate|
+----------+-------+---------+----------+------------+
|        26|   9602|        8|2024-12-09|  2024-12-16|
|        29|  15494|        2|2025-10-29|  2025-10-31|
|       474|  30227|        7|2024-08-28|  2024-08-31|
|       964|  15710|        8|2026-09-04|  2026-09-14|
|      1677|  41724|        1|2025-06-22|  2025-06-27|
|      1697|  48838|        8|2025-12-31|  2026-01-01|
|      1806|  43668|        1|2025-09-02|  2025-09-04|
|      1950|  12723|        8|2024-04-30|  2024-05-01|
|      2040|  49107|        3|2025-03-08|  2025-03-16|
|      2214|  29831|        9|2025-02-02|  2025-02-09|
+----------+-------+---------+----------+------------+
only showing top 10 rows


In [33]:
print("Historical shipments:", historical_shipments.count())
print("API shipments:", new_orders_shipments.count())
print("Combined shipments:", all_shipments.count())

Historical shipments: 40000
API shipments: 300
Combined shipments: 40292


In [34]:
all_shipments.groupBy("ShipmentID") \
    .count() \
    .filter("count > 1") \
    .show()

+----------+-----+
|ShipmentID|count|
+----------+-----+
+----------+-----+



In [35]:
all_shipments.select([
    F.count(
        F.when(F.col(c).isNull(), c)
    ).alias(c)
    for c in all_shipments.columns
]).show()

+----------+-------+---------+--------+------------+
|ShipmentID|OrderID|ShipperID|ShipDate|DeliveryDate|
+----------+-------+---------+--------+------------+
|         1|      1|        1|       1|           1|
+----------+-------+---------+--------+------------+



In [36]:
all_shipments = all_shipments.dropna(how="all")

In [37]:
print("Historical shipments:", historical_shipments.count())
print("API shipments:", new_orders_shipments.count())
print("Combined shipments:", all_shipments.count())

Historical shipments: 40000
API shipments: 300
Combined shipments: 40291


In [38]:
all_orders_details = (
    all_orders_details

    # Make sure types are consistent
    .withColumn("orderdetailid", F.col("orderdetailid").cast("long"))
    .withColumn("orderid", F.col("orderid").cast("long"))
    .withColumn("productid", F.col("productid").cast("long"))
    .withColumn("quantity", F.col("quantity").cast("long"))
    .withColumn("unitprice", F.col("unitprice").cast("decimal(12,2)"))
    .withColumn("discount", F.col("discount").cast("decimal(12,2)"))

    # Handle negative values
    .withColumn(
        "quantity",
        F.when(F.col("quantity") < 0, 0)
         .otherwise(F.col("quantity"))
    )

    .withColumn(
        "unitprice",
        F.when(F.col("unitprice") < 0, 0)
         .otherwise(F.col("unitprice"))
    )

    .withColumn(
        "discount",
        F.when(F.col("discount") < 0, 0)
         .otherwise(F.col("discount"))
    )

    # Recalculate amounts
    .withColumn(
        "gross_amount",
        F.col("quantity") * F.col("unitprice")
    )

    .withColumn(
        "discount_amount",
        F.col("gross_amount") * F.col("discount") / 100
    )

    .withColumn(
        "total_amount",
        F.col("gross_amount") - F.col("discount_amount")
    )
)

In [39]:
all_orders_details.show(2)

+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
|orderdetailid|orderid|productid|quantity|unitprice|discount|gross_amount|discount_amount|total_amount|
+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
|           26|  26004|      342|       5|  1253.86|    0.00|     6269.30|       0.000000| 6269.300000|
|           29|  27162|      707|       3|   521.53|    0.00|     1564.59|       0.000000| 1564.590000|
+-------------+-------+---------+--------+---------+--------+------------+---------------+------------+
only showing top 2 rows


In [40]:
all_orders_details.count()

100732

In [41]:
all_api_orders = (
    api_orders_df
    .withColumn("year", F.year("orderdate"))
    .withColumn("month", F.month("orderdate"))
)
all_api_orders.show(10)

+-------+----------+----------+---------+--------------------+--------------------+-------------+------------+----+-----+
|orderid|customerid| orderdate|   status|            event_id|     event_timestamp|   event_type|      source|year|month|
+-------+----------+----------+---------+--------------------+--------------------+-------------+------------+----+-----+
|2000063|    434201|2026-09-23|CONFIRMED|d2045e20-704c-436...|2026-09-23 02:36:...|ORDER_CREATED|ordering_api|2026|    9|
|2000180|    111243|2026-09-23|CONFIRMED|5013b375-a6bb-4b9...|2026-09-23 02:36:...|ORDER_CREATED|ordering_api|2026|    9|
|2000065|    168318|2026-09-23|CONFIRMED|5a33ba7f-caf8-4e1...|2026-09-23 02:36:...|ORDER_CREATED|ordering_api|2026|    9|
|2000123|    246971|2026-09-23|CONFIRMED|a2e73ccb-075d-4c2...|2026-09-23 02:36:...|ORDER_CREATED|ordering_api|2026|    9|
|2000214|    461816|2026-09-23|CONFIRMED|93811dc9-cd88-45c...|2026-09-23 02:36:...|ORDER_CREATED|ordering_api|2026|    9|
|2000032|    326514|2026

In [42]:
API_CURATED_PATH = f"{PROCESSED_PATH}/api_orders"

(
    all_api_orders
    .write
    .mode("append")
    .format("parquet")
    .partitionBy("year", "month")
    .option("compression", "snappy")
    .save(API_CURATED_PATH)
)


In [43]:
dim_customer = (
    customers
    .select(
        F.col("customerid").cast("long").alias("customer_id"),
        F.trim("firstname").alias("first_name"),
        F.trim("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.trim("firstname"),
            F.trim("lastname")
        ).alias("full_name"),
        F.lower(F.trim("email")).alias("email"),
        F.trim("city").alias("city"),
        F.trim("country").alias("country"),
        F.to_date("registrationdate").alias("registration_date")
    )
    .dropDuplicates(["customer_id"])
)

In [44]:
dim_category = (
    categories
    .select(
        F.col("categoryid").cast("long").alias("category_id"),
        F.trim("categoryname").alias("category_name")
    )
    .dropDuplicates(["category_id"])
)

In [45]:
dim_department = (
    departments
    .select(
        F.col("departmentid").cast("long").alias("department_id"),
        F.trim("departmentname").alias("department_name")
    )
    .dropDuplicates(["department_id"])
)

In [46]:
dim_product = (
    products
    .select(
        F.col("productid").cast("long").alias("product_id"),
        F.col("categoryid").cast("long").alias("category_id"),
        F.trim("productname").alias("product_name"),
        F.trim("brand").alias("brand"),
        F.col("price").cast("decimal(12,2)").alias("price"),
        F.col("cost").cast("decimal(12,2)").alias("cost"),
        F.col("stock").cast("int").alias("stock")
    )
    .dropDuplicates(["product_id"])
)

In [47]:
dim_supplier = (
    suppliers
    .select(
        F.col("supplierid").cast("long").alias("supplier_id"),
        F.trim("suppliername").alias("supplier_name"),
        F.trim("country").alias("country")
    )
    .dropDuplicates(["supplier_id"])
)

In [48]:
dim_employee = (
    employees
    .select(
        F.col("employeeid").cast("long").alias("employee_id"),
        F.col("managerid").cast("long").alias("manager_id"),
        F.col("departmentid").cast("long").alias("department_id"),
        F.trim("firstname").alias("first_name"),
        F.trim("lastname").alias("last_name"),
        F.concat_ws(
            " ",
            F.trim("firstname"),
            F.trim("lastname")
        ).alias("full_name"),
        F.col("salary").cast("decimal(12,2)").alias("salary"),
        F.to_date("hiredate").alias("hire_date")
    )
    .dropDuplicates(["employee_id"])
)

In [49]:
dim_shipper = (
    shippers
    .select(
        F.col("shipperid").cast("long").alias("shipper_id"),
        F.trim("companyname").alias("company_name")
    )
    .dropDuplicates(["shipper_id"])
)

In [50]:
date_range = (
    all_orders
    .select(
        F.to_date("orderdate").alias("date")
    )
    .filter(F.col("date").isNotNull())
    .agg(
        F.min("date").alias("min_date"),
        F.max("date").alias("max_date")
    )
)

In [51]:
dim_date = (
    date_range
    .select(
        F.explode(
            F.sequence(
                F.col("min_date"),
                F.col("max_date"),
                F.expr("interval 1 day")
            )
        ).alias("date")
    )
)

In [52]:
dim_date = (
    dim_date
    .withColumn(
        "date_id",
        F.date_format("date", "yyyyMMdd").cast("int")
    )
    .withColumn("year", F.year("date"))
    .withColumn("quarter", F.quarter("date"))
    .withColumn("month", F.month("date"))
    .withColumn(
        "month_name",
        F.date_format("date", "MMMM")
    )
    .withColumn(
        "day",
        F.dayofmonth("date")
    )
    .withColumn(
        "day_of_week",
        F.dayofweek("date")
    )
    .withColumn(
        "day_name",
        F.date_format("date", "EEEE")
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("date")
    )
)

In [53]:
payment_methods = (
    all_payments
    .select(
        F.upper(
            F.trim("paymentmethod")
        ).alias("payment_method")
    )
    .filter(
        F.col("payment_method").isNotNull()
    )
    .distinct()
)

In [54]:
window_payment = Window.orderBy("payment_method")

dim_payment_method = (
    payment_methods
    .withColumn(
        "payment_method_id",
        F.row_number().over(window_payment)
    )
    .select(
        "payment_method_id",
        "payment_method"
    )
)

In [55]:
order_statuses = (
    all_orders
    .select(
        F.upper(
            F.trim("status")
        ).alias("order_status")
    )
    .filter(
        F.col("order_status").isNotNull()
    )
    .distinct()
)

In [56]:
window_status = Window.orderBy("order_status")

dim_order_status = (
    order_statuses
    .withColumn(
        "order_status_id",
        F.row_number().over(window_status)
    )
    .select(
        "order_status_id",
        "order_status"
    )
)

In [57]:
fact_order = (
    all_orders
    .select(
        F.col("orderid").cast("long").alias("order_id"),
        F.col("customerid").cast("long").alias("customer_id"),
        F.to_date("orderdate").alias("order_date"),
        F.date_format(
            F.to_date("orderdate"),
            "yyyyMMdd"
        ).cast("int").alias("date_id"),
        F.upper(
            F.trim("status")
        ).alias("order_status")
    )
)

In [58]:
fact_order = (
    fact_order
    .join(
        dim_order_status,
        fact_order.order_status ==
        dim_order_status.order_status,
        "left"
    )
    .select(
        "order_id",
        "customer_id",
        "date_id",
        "order_date",
        "order_status_id"
    )
)

In [59]:
fact_order_detail = (
    all_orders_details
    .select(
        F.col("orderdetailid")
            .cast("long")
            .alias("order_detail_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.col("productid")
            .cast("long")
            .alias("product_id"),

        F.col("quantity")
            .cast("int")
            .alias("quantity"),

        F.col("unitprice")
            .cast("decimal(12,2)")
            .alias("unit_price"),

        F.col("discount")
            .cast("decimal(12,2)")
            .alias("discount"),

        F.col("gross_amount")
            .cast("decimal(14,2)")
            .alias("gross_amount"),

        F.col("discount_amount")
            .cast("decimal(14,2)")
            .alias("discount_amount"),

        F.col("total_amount")
            .cast("decimal(14,2)")
            .alias("total_amount")
    )
    .dropDuplicates(["order_detail_id"])
)

In [60]:
fact_order_detail.show(5)

+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+
|order_detail_id|order_id|product_id|quantity|unit_price|discount|gross_amount|discount_amount|total_amount|
+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+
|             26|   26004|       342|       5|   1253.86|    0.00|     6269.30|           0.00|     6269.30|
|             29|   27162|       707|       3|    521.53|    0.00|     1564.59|           0.00|     1564.59|
|            474|   45731|       267|       8|   2340.38|   10.00|    18723.04|        1872.30|    16850.74|
|            964|   13989|       172|       9|   2110.12|   10.00|    18991.08|        1899.11|    17091.97|
|           1677|   48400|       413|       8|   2534.22|    5.00|    20273.76|        1013.69|    19260.07|
+---------------+--------+----------+--------+----------+--------+------------+---------------+------------+
only showing top 5 

In [61]:
fact_payment_base = (
    all_payments
    .select(
        F.col("paymentid")
            .cast("long")
            .alias("payment_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.to_date("paymentdate")
            .alias("payment_date"),

        F.date_format(
            F.to_date("paymentdate"),
            "yyyyMMdd"
        )
        .cast("int")
        .alias("date_id"),

        F.upper(
            F.trim("paymentmethod")
        )
        .alias("payment_method"),

        F.col("amount")
            .cast("decimal(14,2)")
            .alias("amount")
    )
)

In [62]:
fact_payment = (
    fact_payment_base
    .join(
        dim_payment_method,
        fact_payment_base.payment_method ==
        dim_payment_method.payment_method,
        "left"
    )
    .select(
        "payment_id",
        "order_id",
        "date_id",
        "payment_date",
        "payment_method_id",
        "amount"
    )
)

In [63]:
fact_shipment = (
    all_shipments
    .select(
        F.col("shipmentid")
            .cast("long")
            .alias("shipment_id"),

        F.col("orderid")
            .cast("long")
            .alias("order_id"),

        F.col("shipperid")
            .cast("long")
            .alias("shipper_id"),

        F.to_date("shipdate")
            .alias("ship_date"),

        F.to_date("deliverydate")
            .alias("delivery_date")
    )
    .withColumn(
        "shipping_days",
        F.datediff(
            F.col("delivery_date"),
            F.col("ship_date")
        )
    )
)

In [64]:
fact_shipment.show(2)

+-----------+--------+----------+----------+-------------+-------------+
|shipment_id|order_id|shipper_id| ship_date|delivery_date|shipping_days|
+-----------+--------+----------+----------+-------------+-------------+
|         26|    9602|         8|2024-12-09|   2024-12-16|            7|
|         29|   15494|         2|2025-10-29|   2025-10-31|            2|
+-----------+--------+----------+----------+-------------+-------------+
only showing top 2 rows


In [65]:
customer_sales_base = (
    fact_order_detail
    .join(
        fact_order.select(
            "order_id",
            "customer_id",
            "order_date"
        ),
        "order_id",
        "inner"
    )
)

In [66]:
fact_customer_sales = (
    customer_sales_base
    .groupBy("customer_id")
    .agg(
        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("total_quantity"),

        F.sum(
            "gross_amount"
        ).alias("gross_sales"),

        F.sum(
            "discount_amount"
        ).alias("total_discount"),

        F.sum(
            "total_amount"
        ).alias("total_sales"),

        F.avg(
            "total_amount"
        ).alias("average_order_line"),

        F.min(
            "order_date"
        ).alias("first_order_date"),

        F.max(
            "order_date"
        ).alias("last_order_date")
    )
)

In [67]:
fact_customer_sales = (
    fact_customer_sales
    .withColumn(
        "customer_segment",
        F.when(
            F.col("total_sales") >= 10000,
            "VIP"
        )
        .when(
            F.col("total_sales") >= 5000,
            "PREMIUM"
        )
        .when(
            F.col("total_sales") >= 1000,
            "REGULAR"
        )
        .otherwise(
            "LOW_VALUE"
        )
    )
)

In [68]:
fact_customer_sales.show(1)

+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
|customer_id|total_orders|total_quantity|gross_sales|total_discount|total_sales|average_order_line|first_order_date|last_order_date|customer_segment|
+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
|       5385|           5|            70|  119659.94|      10581.10|  109078.85|       6817.428125|      2024-02-03|     2026-03-05|             VIP|
+-----------+------------+--------------+-----------+--------------+-----------+------------------+----------------+---------------+----------------+
only showing top 1 row


In [69]:
fact_product_sales = (
    fact_order_detail
    .groupBy("product_id")
    .agg(
        F.countDistinct(
            "order_id"
        ).alias("total_orders"),

        F.sum(
            "quantity"
        ).alias("units_sold"),

        F.sum(
            "gross_amount"
        ).alias("gross_sales"),

        F.sum(
            "discount_amount"
        ).alias("total_discount"),

        F.sum(
            "total_amount"
        ).alias("total_sales"),

        F.avg(
            "unit_price"
        ).alias("average_unit_price")
    )
)

In [70]:
bridge_product_supplier = (
    product_suppliers
    .select(
        F.col("productid")
            .cast("long")
            .alias("product_id"),

        F.col("supplierid")
            .cast("long")
            .alias("supplier_id")
    )
    .dropDuplicates(
        ["product_id", "supplier_id"]
    )
)

In [71]:
write_parquet(dim_customer, "dim_customer")
write_parquet(dim_product, "dim_product")
write_parquet(dim_category, "dim_category")
write_parquet(dim_department, "dim_department")
write_parquet(dim_supplier, "dim_supplier")
write_parquet(dim_employee, "dim_employee")
write_parquet(dim_shipper, "dim_shipper")
write_parquet(dim_date, "dim_date")
write_parquet(dim_payment_method, "dim_payment_method")
write_parquet(dim_order_status, "dim_order_status")

Writing dim_customer -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_customer
Writing dim_product -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_product
Writing dim_category -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_category
Writing dim_department -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_department
Writing dim_supplier -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_supplier
Writing dim_employee -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_employee
Writing dim_shipper -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_shipper
Writing dim_date -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_date
Writing dim_payment_method -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/dim_payment_method
Writing dim_order_status -> s3a:/

In [72]:
write_parquet(fact_order, "fact_order")
write_parquet(fact_order_detail, "fact_order_detail")
write_parquet(fact_payment, "fact_payment")
write_parquet(fact_shipment, "fact_shipment")
write_parquet(fact_customer_sales, "fact_customer_sales")
write_parquet(fact_product_sales, "fact_product_sales")

Writing fact_order -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/fact_order
Writing fact_order_detail -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/fact_order_detail
Writing fact_payment -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/fact_payment
Writing fact_shipment -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/fact_shipment
Writing fact_customer_sales -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/fact_customer_sales
Writing fact_product_sales -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/fact_product_sales


In [73]:
write_parquet(bridge_product_supplier,"bridge_product_supplier")

Writing bridge_product_supplier -> s3a://yassin-s3-bucket-161828268081-eu-north-1-an/processed/ecommerce/bridge_product_supplier
